<a href="https://colab.research.google.com/github/Solmaeir/Roman_Columns/blob/main/Roman_Columns.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Roma Dönemi Kolon Restorasyonu
**Görüntü İşleme Projesi — v10**

### Pipeline
1. Bilateral Filter + CLAHE ön işleme  
2. SAM segmentasyonu (→ Klasik CV fallback)  
3. **Yapı analizi**: Capital/base/shaft tespiti + kolon ekseni  
4. **Kenar kesim tespiti**: Yukarıdan aşağı tarar; sol/sağ kenarı bağımsız izler; her ikisi de sağlam olan son satır = kesim noktası  
5. **Eksen hizalı doldurma**: Kırık bölge + uzatma tamamen yeniden üretilir (kesim noktasına yakın doku kaynağı)  
6. **Dolgu iyileştirme**: Şaft sınırına kırpma + kenar yumuşatma + seam harmanlama  
7. **Yatay simetri**: Yalnızca sol/sağ hasar varsa uygula  
8. Önce / Sonra görselleştirme

In [ ]:
# ── Google Drive bağla + klasör yolları ──────────────────────────────────────
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB      = True
    BASE          = '/content/drive/MyDrive/sutun_veriseti_1'
    DAMAGED_DIR   = os.path.join(BASE, 'damaged_column')
    REFERENCE_DIR = os.path.join(BASE, 'referance_column')
    print('Google Colab — Drive bağlandı.')
except ImportError:
    IN_COLAB      = False
    BASE          = '.'
    DAMAGED_DIR   = os.path.join(BASE, 'damaged_column')
    REFERENCE_DIR = os.path.join(BASE, 'referance_column')
    print('Yerel ortam.')

damaged_files   = [f for f in os.listdir(DAMAGED_DIR)
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
reference_files = [f for f in os.listdir(REFERENCE_DIR)
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

print(f'Damaged  : {len(damaged_files)} görüntü')
print(f'Reference: {len(reference_files)} görüntü')
print(f'Test [25]: {damaged_files[25]}')

In [ ]:
# ── Bağımlılıklar ──────────────────────────────────────────────────────────────────
import subprocess, sys

def pip_q(pkg):
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=False)

try:
    import segment_anything; print('segment-anything OK')
except ImportError:
    pip_q('segment-anything')

try:
    import simple_lama_inpainting; print('simple-lama-inpainting OK')
except ImportError:
    pip_q('simple-lama-inpainting')

CKPT = 'sam_vit_b_01ec64.pth'
if not os.path.exists(CKPT):
    print('SAM checkpoint indiriliyor…')
    subprocess.run(['wget', '-q',
        'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth'], check=False)
    print('OK')
else:
    print('SAM checkpoint mevcut.')

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import torch
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

print(f'NumPy  : {np.__version__}')
print(f'OpenCV : {cv2.__version__}')
print(f'Torch  : {torch.__version__}')
print(f'PIL    : {Image.__version__}')

SAM_AVAILABLE = False
try:
    from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
    SAM_AVAILABLE = True
    print(f'SAM    : OK  (CUDA={torch.cuda.is_available()})')
except Exception as e:
    print(f'SAM    : Yok — {e}')

LAMA_AVAILABLE = False
try:
    from simple_lama_inpainting import SimpleLama
    LAMA_AVAILABLE = True
    print('LaMa   : OK')
except Exception as e:
    print(f'LaMa   : Yok — {e}')

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

mask_generator = None
if SAM_AVAILABLE:
    try:
        _sam = sam_model_registry['vit_b'](checkpoint=CKPT)
        _sam.to(DEVICE)
        mask_generator = SamAutomaticMaskGenerator(
            model=_sam,
            points_per_side=32,
            pred_iou_thresh=0.86,
            stability_score_thresh=0.90,
            min_mask_region_area=300,
        )
        print(f'SAM yüklendi ({DEVICE}).')
    except Exception as e:
        print(f'SAM yüklenemedi: {e}')
        SAM_AVAILABLE = False

lama_model = None
if LAMA_AVAILABLE:
    try:
        lama_model = SimpleLama()
        print('LaMa yüklendi.')
    except Exception as e:
        print(f'LaMa yüklenemedi: {e}')
        LAMA_AVAILABLE = False

if not SAM_AVAILABLE:   print('→ Klasik CV aktif.')
if not LAMA_AVAILABLE:  print('→ OpenCV TELEA aktif.')

In [ ]:
def preprocess(img_bgr):
    dn = cv2.bilateralFilter(img_bgr, d=9, sigmaColor=75, sigmaSpace=75)
    lab = cv2.cvtColor(dn, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return cv2.cvtColor(cv2.merge([clahe.apply(l), a, b]), cv2.COLOR_LAB2BGR)

def column_score(cnt, h_img, w_img):
    area = cv2.contourArea(cnt)
    if area < 500: return 0.0
    x, y, w, h = cv2.boundingRect(cnt)
    asp = h / (w + 1e-5)
    ar  = area / (h_img * w_img)
    return asp * ar if asp > 1.5 and 0.03 < ar < 0.80 else 0.0

def detect_column_classical(img_bgr):
    h_img, w_img = img_bgr.shape[:2]
    gray    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    adapt   = cv2.adaptiveThreshold(blurred, 255,
                  cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 31, 5)
    edges   = cv2.Canny(blurred, 20, 80)
    combo   = cv2.bitwise_or(adapt, edges)
    kr = cv2.getStructuringElement(cv2.MORPH_RECT,   (5, 5))
    ke = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(9, 9))
    closed  = cv2.morphologyEx(combo,   cv2.MORPH_CLOSE, kr, iterations=4)
    dilated = cv2.dilate(closed, kr, iterations=2)
    cnts, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts: return None
    scored = sorted(cnts, key=lambda c: column_score(c, h_img, w_img), reverse=True)
    best   = scored[0]
    mask   = np.zeros((h_img, w_img), dtype=np.uint8)
    cv2.drawContours(mask, [best], -1, 255, -1)
    return cv2.morphologyEx(mask, cv2.MORPH_CLOSE, ke, iterations=5)

def detect_column_sam(img_bgr):
    if mask_generator is None: return None
    h_img, w_img = img_bgr.shape[:2]
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    try:
        masks = mask_generator.generate(img_rgb)
    except Exception as e:
        print(f'  SAM hatası: {e}'); return None
    if not masks: return None
    ms = sorted(masks, key=lambda m: m['area'], reverse=True)
    best, best_s = None, 0.0
    for m in ms[:12]:
        seg = m['segmentation'].astype(np.uint8) * 255
        cnts, _ = cv2.findContours(seg, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not cnts: continue
        s = column_score(max(cnts, key=cv2.contourArea), h_img, w_img)
        if s > best_s: best_s, best = s, seg
    if best is None:
        best = ms[0]['segmentation'].astype(np.uint8) * 255
    ke = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    return cv2.morphologyEx(best, cv2.MORPH_CLOSE, ke, iterations=4)

def detect_column(img_bgr):
    if SAM_AVAILABLE:
        m = detect_column_sam(img_bgr)
        if m is not None:
            print('  SAM ile tespit edildi.'); return m, 'SAM'
    m = detect_column_classical(img_bgr)
    if m is not None:
        print('  Klasik CV ile tespit edildi.'); return m, 'Klasik CV'
    return None, None

def get_bbox(mask):
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts: return None
    cnt = max(cnts, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(cnt)
    return x, y, w, h, x + w // 2, cnt

print('Yardımcı fonksiyonlar hazır.')

In [ ]:
# ── Kolon yapısı analizi + kenar tabanlı kesim tespiti ───────────────────────────────────────────

def _moving_avg(arr, k=9):
    kern = np.ones(k, dtype=np.float32) / k
    return np.convolve(arr.astype(np.float32), kern, mode='same')


def analyze_column_structure(col_mask, capital_ratio=1.12):
    h_img, w_img = col_mask.shape

    widths  = np.zeros(h_img, dtype=np.float32)
    centers = np.zeros(h_img, dtype=np.float32)
    for y in range(h_img):
        nz = np.where(col_mask[y, :] > 128)[0]
        if len(nz) > 2:
            widths[y]  = nz[-1] - nz[0] + 1
            centers[y] = (nz[-1] + nz[0]) / 2.0

    active = np.where(widths > 0)[0]
    if len(active) < 20:
        return None

    col_top    = int(active[0])
    col_bottom = int(active[-1])
    col_h      = col_bottom - col_top

    q1 = col_top + col_h // 4
    q3 = col_top + 3 * col_h // 4
    mid_w = widths[q1:q3]
    mid_w = mid_w[mid_w > 0]
    if len(mid_w) == 0:
        return None
    shaft_width = float(np.median(mid_w))
    threshold   = shaft_width * capital_ratio

    w_smooth = _moving_avg(widths, k=9)

    capital_end = col_top
    in_cap = False
    for y in range(col_top, col_top + col_h // 2):
        if w_smooth[y] > threshold:
            capital_end = y; in_cap = True
        elif in_cap and w_smooth[y] > 0:
            break
    has_capital = in_cap and (capital_end - col_top) > col_h * 0.04

    base_start = col_bottom
    in_base = False
    for y in range(col_bottom, col_top + col_h // 2, -1):
        if w_smooth[y] > threshold:
            base_start = y; in_base = True
        elif in_base and w_smooth[y] > 0:
            break
    has_base = in_base and (col_bottom - base_start) > col_h * 0.04

    shaft_top    = capital_end + 1 if has_capital else col_top
    shaft_bottom = base_start  - 1 if has_base    else col_bottom

    broken_thresh = shaft_width * 0.75
    clean_bot = shaft_bottom
    for y in range(shaft_bottom, shaft_top, -1):
        if widths[y] >= broken_thresh:
            clean_bot = y; break
    clean_top = shaft_top
    for y in range(shaft_top, shaft_bottom):
        if widths[y] >= broken_thresh:
            clean_top = y; break

    shaft_rows = [y for y in range(clean_top, clean_bot + 1)
                  if widths[y] >= broken_thresh]
    if len(shaft_rows) > 10:
        yy = np.array(shaft_rows, dtype=np.float64)
        cc = np.array([centers[y] for y in shaft_rows], dtype=np.float64)
        coeffs         = np.polyfit(yy, cc, 1)
        axis_slope     = float(coeffs[0])
        axis_intercept = float(coeffs[1])
    else:
        info           = get_bbox(col_mask)
        axis_slope     = 0.0
        axis_intercept = float(info[4]) if info else w_img / 2.0

    print(f'  Capital : {has_capital}  (y ≤ {capital_end})')
    print(f'  Base    : {has_base}  (y ≥ {base_start})')
    print(f'  Şaft    : y=[{shaft_top}…{shaft_bottom}]  '
          f'Temiz uç=[{clean_top}…{clean_bot}]  '
          f'Genişlik≈{shaft_width:.0f}px')
    print(f'  Eksen   : eğim={axis_slope:.4f}')

    return dict(
        col_top=col_top, col_bottom=col_bottom,
        capital_end=capital_end, base_start=base_start,
        shaft_top=shaft_top, shaft_bottom=shaft_bottom,
        clean_shaft_top=clean_top, clean_shaft_bottom=clean_bot,
        shaft_width=shaft_width,
        has_capital=has_capital, has_base=has_base,
        axis_slope=axis_slope, axis_intercept=axis_intercept,
        widths=widths, centers=centers,
    )


def find_edge_cut(col_mask, struct, thresh_ratio=0.08):
    """
    Şaftı YUKARIDAN AŞAĞI tara.
    Sol ve sağ kenarları bağımsız izle:
      - Her satırda beklenen konum: eksen merkezi ± shaft_width/2
      - Gerçek kenar beklenen konuma yeterince yakınsa → o kenar o satırda sağlam
    Her iki kenarın birlikte sağlam olduğu en son satır = cut_y.
    (Kullanıcının çizdiği: mavi=kısa kenar, kırmızı=uzun kenar, sarı=mavi bitişi)
    """
    cst_top   = struct['clean_shaft_top']
    cst_bot   = struct['clean_shaft_bottom']
    st_w      = int(round(struct['shaft_width']))
    half      = st_w // 2
    slope     = struct['axis_slope']
    intercept = struct['axis_intercept']
    thresh    = st_w * thresh_ratio

    last_left_y  = cst_top
    last_right_y = cst_top

    for y in range(cst_top, cst_bot + 1):
        nz = np.where(col_mask[y, :] > 128)[0]
        if len(nz) < 3:
            continue
        cx        = slope * y + intercept
        act_left  = float(nz[0])
        act_right = float(nz[-1])

        if abs(act_left  - (cx - half)) <= thresh:
            last_left_y  = y
        if abs(act_right - (cx + half)) <= thresh:
            last_right_y = y

    cut_y = min(last_left_y, last_right_y)
    side  = 'sol (mavi)' if last_left_y <= last_right_y else 'sağ (mavi)'
    print(f'  Sol kenar son temiz satır : y={last_left_y}')
    print(f'  Sağ kenar son temiz satır : y={last_right_y}')
    print(f'  Kesim (sarı çizgi)        : y={cut_y}  ({side} kenar önce bitiyor)')
    return cut_y


def side_damage_ratio(col_mask):
    info = get_bbox(col_mask)
    if info is None: return 0.0, 0.0
    x, y, w, h, cx, _ = info
    ideal = np.zeros_like(col_mask)
    ideal[y:y+h, x:x+w] = 255
    def r(ri, ra):
        d = np.sum(ri > 0)
        return max(0.0, (d - np.sum(ra > 0)) / d) if d else 0.0
    return (r(ideal[y:y+h, x:cx],   col_mask[y:y+h, x:cx]),
            r(ideal[y:y+h, cx:x+w], col_mask[y:y+h, cx:x+w]))


print('analyze_column_structure() + find_edge_cut() hazır.')


In [ ]:
# ── Eksen hizalı şaft uzatma (agresif bbox temizleme) ────────────────────────

def extend_shaft_axis_aligned(img_bgr, col_mask, struct,
                               extra_ratio=0.55, direction='auto'):
    """
    Kolon eksenini takip ederek şaftı uzat.

    AGRESİF MOD:
      - cut_y altındaki TÜM bbox alanı arka plan rengiyle temizlenir.
      - Sonra cut_y'den target'a kadar şaft dokusu yerleştirilir.
      - Doku kaynağı: kesim noktasına en yakın %40 kullanılır →
        kolon çizgisi geçişi daha pürüssüz olur.
    """
    h_img, w_img = img_bgr.shape[:2]

    has_cap   = struct['has_capital']
    has_base  = struct['has_base']
    cst_top   = struct['clean_shaft_top']
    cst_bot   = struct['clean_shaft_bottom']
    col_bot   = struct['col_bottom']
    st_w      = max(4, int(round(struct['shaft_width'])))
    slope     = struct['axis_slope']
    intercept = struct['axis_intercept']

    if direction == 'auto':
        dirs = []
        if not has_base: dirs.append('bottom')
        if not has_cap:  dirs.append('top')
        if not dirs:     dirs = ['bottom']
    elif direction == 'both':
        dirs = ['bottom', 'top']
    else:
        dirs = [direction]

    shaft_h = cst_bot - cst_top
    if shaft_h < 20:
        print('  Şaft çok kısa, uzatma yapılamıyor.')
        return img_bgr.copy(), \
               np.zeros((h_img, w_img), dtype=np.uint8), \
               np.zeros((h_img, w_img), dtype=np.uint8)

    bg_px    = img_bgr[col_mask == 0]
    bg_color = (np.median(bg_px.reshape(-1, 3), axis=0).astype(np.uint8)
                if len(bg_px) > 100 else np.array([150, 150, 150], dtype=np.uint8))

    cnts, _ = cv2.findContours(col_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if cnts:
        cnt    = max(cnts, key=cv2.contourArea)
        bx, by, bw, bh = cv2.boundingRect(cnt)
    else:
        bx, by, bw, bh = 0, 0, w_img, h_img
    margin_x = max(15, st_w // 3)
    clear_x0 = max(0, bx - margin_x)
    clear_x1 = min(w_img, bx + bw + margin_x)

    result    = img_bgr.copy()
    ext_mask  = np.zeros((h_img, w_img), dtype=np.uint8)
    seam_mask = np.zeros((h_img, w_img), dtype=np.uint8)

    def _place_row(fy, src_y):
        dcx  = int(round(slope * fy  + intercept))
        half = st_w // 2
        dx0  = max(0, dcx - half);  dx1 = min(w_img, dx0 + st_w)
        if dx1 <= dx0: return
        scx  = int(round(slope * src_y + intercept))
        sx0  = max(0, scx - half);  sx1 = min(w_img, sx0 + st_w)
        src  = img_bgr[src_y, sx0:sx1].copy()
        sw   = sx1 - sx0;  dw = dx1 - dx0
        if sw <= 0: return
        if sw != dw:
            src = cv2.resize(src.reshape(1, sw, 3),
                             (dw, 1), interpolation=cv2.INTER_LINEAR).reshape(dw, 3)
        result[fy, dx0:dx1]   = src
        ext_mask[fy, dx0:dx1] = 255

    for d in dirs:
        if d == 'bottom':
            extend_px  = int(shaft_h * extra_ratio)
            min_target = col_bot + int(shaft_h * 0.30)
            target     = min(h_img - 5, max(cst_bot + extend_px, min_target))

            clear_y0 = cst_bot + 1
            clear_y1 = min(h_img, target + 1)
            result[clear_y0:clear_y1, clear_x0:clear_x1] = bg_color
            print(f'  Agresif temizleme: y=[{clear_y0}…{clear_y1-1}] '
                  f'x=[{clear_x0}…{clear_x1-1}]')

            # Kesim noktasına yakın alt %40 → seam süreklilik en iyi
            tex_src_h = max(10, int(shaft_h * 0.40))
            tex_y0    = max(cst_top, cst_bot - tex_src_h)
            tex_y1    = cst_bot
            tex_seq   = list(range(tex_y0, tex_y1)) or [max(cst_top, cst_bot - 1)]
            n_tex     = len(tex_seq)
            fill_range = range(clear_y0, clear_y1)
            sy0, sy1   = max(0, cst_bot - 25), min(h_img, cst_bot + 25)
            print(f'  Alt uzatma : {cst_bot} → {target} px  '
                  f'(kırık={col_bot - cst_bot}px  yeni={target - col_bot}px)')
        else:
            extend_px  = int(shaft_h * extra_ratio)
            target     = max(5, cst_top - extend_px)

            clear_y0 = max(0, target)
            clear_y1 = cst_top
            if clear_y1 > clear_y0:
                result[clear_y0:clear_y1, clear_x0:clear_x1] = bg_color

            # Kesim noktasına yakın üst %40 (tersine)
            tex_src_h = max(10, int(shaft_h * 0.40))
            tex_y0    = cst_top
            tex_y1    = min(cst_bot, cst_top + tex_src_h)
            tex_seq   = list(reversed(range(tex_y0, tex_y1))) or [min(cst_bot - 1, cst_top + 1)]
            n_tex     = len(tex_seq)
            fill_range = range(cst_top - 1, target - 1, -1)
            sy0, sy1   = max(0, cst_top - 25), min(h_img, cst_top + 25)
            print(f'  Üst uzatma : {cst_top} → {target} px  '
                  f'(+{cst_top - target} px)')

        for i, fy in enumerate(fill_range):
            _place_row(fy, tex_seq[i % n_tex])

        band = ext_mask[sy0:sy1, :]
        seam_mask[sy0:sy1, :] = np.where(band > 0, 255, seam_mask[sy0:sy1, :])

    return result, ext_mask, seam_mask


print('extend_shaft_axis_aligned() hazır.')


In [ ]:
# ── Dolgu sonrası iyileştirme ─────────────────────────────────────────────────

def post_process_fill(result_bgr, orig_bgr, ext_mask, struct,
                      seam_blend=18, edge_feather=5):
    """
    Eksen hizalı dolgudan sonra kalite iyileştirme:
    1. Şaft sınırına kırpma     → taşan pikseller (overflow) temizlenir
    2. Sol/sağ kenar feathering → pürüzlü sınırlar yumuşatılır
    3. Kesim noktasında alfa-harmanlama → kolon çizgisi süreklilik sağlanır
    4. Bilateral filtre         → dolgu bölgesinde genel yumuşatma
    """
    h_img, w_img = result_bgr.shape[:2]
    out = result_bgr.copy()

    cst_bot   = struct['clean_shaft_bottom']
    slope     = struct['axis_slope']
    intercept = struct['axis_intercept']
    st_w      = max(4, int(round(struct['shaft_width'])))
    half      = st_w // 2

    fill_rows = np.where(np.any(ext_mask > 0, axis=1))[0]
    if len(fill_rows) == 0:
        return out

    bg_px    = orig_bgr[ext_mask == 0]
    bg_color = (np.median(bg_px.reshape(-1, 3), axis=0).astype(np.float32)
                if len(bg_px) > 100 else np.array([150, 150, 150], dtype=np.float32))

    # 1. Şaft sınır maskesi oluştur ve taşanları temizle
    shaft_fill_mask = np.zeros((h_img, w_img), dtype=np.uint8)
    for y in fill_rows:
        cx  = int(round(slope * y + intercept))
        x0  = max(0, cx - half)
        x1  = min(w_img, cx + half)
        shaft_fill_mask[y, x0:x1] = 255

    overflow = cv2.bitwise_and(ext_mask, cv2.bitwise_not(shaft_fill_mask))
    out[overflow > 0] = bg_color.astype(np.uint8)

    # 2. Sol/sağ kenar feathering — sınırdaki piksel sertliğini gider
    for y in fill_rows:
        cx  = int(round(slope * y + intercept))
        x0  = max(0, cx - half)
        x1  = min(w_img, cx + half)
        row = out[y].astype(np.float32)
        for dx in range(min(edge_feather, half)):
            alpha = dx / edge_feather         # 0=arka plan, 1=dolgu
            px_l  = x0 + dx
            px_r  = x1 - 1 - dx
            if 0 <= px_l < w_img:
                row[px_l] = (1 - alpha) * bg_color + alpha * row[px_l]
            if 0 <= px_r < w_img:
                row[px_r] = (1 - alpha) * bg_color + alpha * row[px_r]
        out[y] = np.clip(row, 0, 255).astype(np.uint8)

    # 3. Kesim çizgisinde alfa-harmanlama (orijinal ↔ dolgu)
    seam_t = max(0,     cst_bot - seam_blend // 3)
    seam_b = min(h_img, cst_bot + seam_blend + 1)
    for y in range(seam_t, seam_b):
        cx  = int(round(slope * y + intercept))
        x0  = max(0, cx - half); x1 = min(w_img, cx + half)
        if x1 <= x0: continue
        alpha = max(0.0, min(1.0, (y - seam_t) / max(1, seam_b - seam_t - 1)))
        s = slice(x0, x1)
        blended = ((1 - alpha) * orig_bgr[y, s].astype(np.float32) +
                    alpha * result_bgr[y, s].astype(np.float32))
        out[y, s] = np.clip(blended, 0, 255).astype(np.uint8)

    # 4. Bilateral filtre — dolgu bölgesi + küçük çevre
    ke     = np.ones((3, 3), np.uint8)
    region = cv2.dilate(shaft_fill_mask, ke, iterations=2)
    smooth = cv2.bilateralFilter(out, d=9, sigmaColor=40, sigmaSpace=40)
    out[region > 0] = smooth[region > 0]

    return out


print('post_process_fill() hazır.')


In [ ]:
def symmetry_restore(img_bgr, col_mask):
    h_img, w_img = img_bgr.shape[:2]
    info = get_bbox(col_mask)
    if info is None:
        return img_bgr.copy(), None
    x, y, w, h, cx, _ = info
    left_px  = np.sum(col_mask[:, x:cx]    > 128)
    right_px = np.sum(col_mask[:, cx:x + w] > 128)
    result = img_bgr.copy()
    if left_px >= right_px:
        good   = img_bgr[:, x:cx].copy()
        mirror = cv2.flip(good, 1)
        mw     = mirror.shape[1]
        xs, xe = cx, min(w_img, cx + mw)
        aw     = xe - xs
        empty  = col_mask[:, xs:xe] == 0
        tmp    = result[:, xs:xe].copy()
        tmp[empty] = mirror[:, :aw][empty]
        result[:, xs:xe] = tmp
        side = 'sağ (sol referans)'
    else:
        good   = img_bgr[:, cx:x + w].copy()
        mirror = cv2.flip(good, 1)
        mw     = mirror.shape[1]
        xs, xe = max(0, cx - mw), cx
        aw     = xe - xs
        empty  = col_mask[:, xs:xe] == 0
        tmp    = result[:, xs:xe].copy()
        tmp[empty] = mirror[:, mw - aw:][empty]
        result[:, xs:xe] = tmp
        side = 'sol (sağ referans)'
    print(f'  Simetri: {side} | Sol={left_px}px Sağ={right_px}px')
    return result, cx

print('symmetry_restore() hazır.')

In [ ]:
def inpaint(img_bgr, mask):
    if np.sum(mask) < 50:
        return img_bgr.copy()
    if LAMA_AVAILABLE and lama_model is not None:
        try:
            img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            res_pil  = lama_model(Image.fromarray(img_rgb), Image.fromarray(mask))
            return cv2.cvtColor(np.array(res_pil), cv2.COLOR_RGB2BGR)
        except Exception as e:
            print(f'  LaMa hatası: {e} → TELEA')
    return cv2.inpaint(img_bgr, mask, inpaintRadius=7, flags=cv2.INPAINT_TELEA)

print('inpaint() hazır.')

In [ ]:
# ── Ana Pipeline ─────────────────────────────────────────────────────────────

def restore_column(
    img_path,
    extra_ratio=0.55,
    sym_threshold=0.13,
    direction='auto',
    capital_ratio=1.12,
    edge_thresh=0.08,
    force_cut_ratio=0.30,
):
    img = cv2.imread(img_path)
    if img is None:
        raise FileNotFoundError(f'Görüntü okunamadı: {img_path}')

    fname = os.path.basename(img_path)
    H, W  = img.shape[:2]
    print(f"\n{'='*65}")
    print(f'  {fname}  ({W}×{H} px)')
    print(f"{'='*65}")

    print('① Ön işleme…')
    proc = preprocess(img)

    print('② Segmentasyon…')
    col_mask, method = detect_column(proc)
    if col_mask is None:
        print('HATA: Kolon tespit edilemedi.'); return None

    print('③ Yapı analizi…')
    struct = analyze_column_structure(col_mask, capital_ratio=capital_ratio)
    if struct is None:
        print('HATA: Yapı analizi başarısız.'); return None

    print('④ Kenar kesim noktası…')
    cut_y = find_edge_cut(col_mask, struct, thresh_ratio=edge_thresh)

    # Güvenlik ağı: cut_y col_bottom'a çok yakınsa görünür kırık temizlenemez.
    shaft_h_full = struct['clean_shaft_bottom'] - struct['clean_shaft_top']
    if shaft_h_full < 20:
        shaft_h_full = struct['col_bottom'] - struct['col_top']
    min_broken_height = max(20, int(shaft_h_full * force_cut_ratio))
    max_allowed_cut   = struct['col_bottom'] - min_broken_height
    if cut_y > max_allowed_cut:
        print(f'  ⚠ Cut çok aşağıda ({cut_y}), zorla yukarı: → {max_allowed_cut}')
        cut_y = max_allowed_cut

    if cut_y < struct['clean_shaft_bottom']:
        print(f'  clean_shaft_bottom: {struct["clean_shaft_bottom"]} → {cut_y}')
        struct['clean_shaft_bottom'] = cut_y

    actual_cut      = struct['clean_shaft_bottom']
    col_bottom_orig = struct['col_bottom']

    current   = proc.copy()
    all_masks = []
    steps_log = []

    print('⑤ Eksen hizalı uzatma (agresif bbox temizleme)…')
    ext_bgr, ext_mask, seam_mask = extend_shaft_axis_aligned(
        current, col_mask, struct,
        extra_ratio=extra_ratio, direction=direction
    )
    if np.sum(ext_mask) > 0:
        ext_bgr = inpaint(ext_bgr, seam_mask)
        print('⑥ Dolgu iyileştirme (kırpma + kenar yumuşatma + seam harmanlama)…')
        ext_bgr = post_process_fill(ext_bgr, current, ext_mask, struct)
        current = ext_bgr
        all_masks.append(('Uzatma', ext_mask))
        steps_log.append('uzatma')
    else:
        print('  Uzatma uygulanmadı.')

    ldmg, rdmg = side_damage_ratio(col_mask)
    print(f'⑦ Sol/sağ hasar: sol={ldmg:.2f}  sağ={rdmg:.2f}  (eşik={sym_threshold})')
    if max(ldmg, rdmg) > sym_threshold:
        print('  Yatay simetri uygulanıyor…')
        sym_bgr, sym_axis = symmetry_restore(current, col_mask)
        info = get_bbox(col_mask)
        x, y, w, h, cx, _ = info
        ideal = np.zeros_like(col_mask)
        ideal[y:y+h, x:x+w] = 255
        ke    = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        col_f = cv2.morphologyEx(col_mask, cv2.MORPH_CLOSE, ke, iterations=3)
        gap   = cv2.bitwise_and(ideal, cv2.bitwise_not(col_f))
        gap   = cv2.morphologyEx(gap, cv2.MORPH_OPEN, ke, iterations=1)
        if np.sum(gap) > 200:
            sym_bgr = inpaint(sym_bgr, gap)
        current = sym_bgr
        all_masks.append(('Simetri', gap))
        steps_log.append('simetri')

    if not steps_log:
        print('  ⚠ Hiçbir adım tetiklenmedi.')

    final_bgr = cv2.bilateralFilter(current, d=7, sigmaColor=50, sigmaSpace=50)

    img_rgb   = cv2.cvtColor(img,       cv2.COLOR_BGR2RGB)
    final_rgb = cv2.cvtColor(final_bgr, cv2.COLOR_BGR2RGB)

    vis_struct = img_rgb.copy()
    sl, ic = struct['axis_slope'], struct['axis_intercept']
    for yy in range(H):
        xx = int(round(sl * yy + ic))
        if 0 <= xx < W:
            vis_struct[yy, xx] = [255, 50, 50]
    if struct['has_capital']:
        cv2.line(vis_struct, (0, struct['capital_end']),
                 (W, struct['capital_end']), (50, 200, 50), 2)
    cv2.line(vis_struct, (0, actual_cut),
             (W, actual_cut), (255, 255, 0), 2)

    combined_mask = np.zeros((H, W), dtype=np.uint8)
    for _, m in all_masks:
        combined_mask = cv2.bitwise_or(combined_mask, m)

    fig, axes = plt.subplots(1, 5, figsize=(28, 8))
    axes[0].imshow(img_rgb)
    axes[0].set_title('① Orijinal', fontsize=12, fontweight='bold', color='darkred')
    axes[1].imshow(col_mask, cmap='gray')
    axes[1].set_title(f'② Maske ({method})', fontsize=11)
    axes[2].imshow(vis_struct)
    axes[2].set_title('③+④ Yapı & Kesim
Yeşil=Capital  Sarı=Kesim
Kırmızı=Eksen', fontsize=10)
    axes[3].imshow(combined_mask, cmap='hot')
    axes[3].set_title('⑤ Doldurulan Bölge
(kırık+uzatma)', fontsize=11)
    axes[4].imshow(final_rgb)
    axes[4].set_title('⑥ RESTORE EDİLMİŞ', fontsize=12, fontweight='bold', color='darkgreen')

    for ax in axes: ax.axis('off')
    methods_str = ' + '.join(steps_log) if steps_log else 'yok'
    plt.suptitle(f'Roma Kolon Restorasyonu — {fname}\n[{methods_str}]', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('restoration_steps.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("→ 'restoration_steps.png' kaydedildi.")

    return dict(original=img_rgb, final=final_rgb,
                col_mask=col_mask, combined_mask=combined_mask,
                struct=struct, steps=steps_log)


print('restore_column() hazır.')


In [ ]:
damaged_files = [f for f in os.listdir(DAMAGED_DIR)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

test_file = damaged_files[25]
test_path = os.path.join(DAMAGED_DIR, test_file)
print(f'Test: [25] {test_file}')

result = restore_column(
    test_path,
    extra_ratio=0.55,
    sym_threshold=0.13,
    direction='auto',
    capital_ratio=1.12,
    edge_thresh=0.08,
)


In [ ]:
if result is not None:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 14))
    ax1.imshow(result['original'])
    ax1.set_title('ÖNCE\n(Hasalı Kolon)', fontsize=18, fontweight='bold', color='darkred')
    ax1.axis('off')
    ax2.imshow(result['final'])
    ax2.set_title('SONRA\n(Restore Edilmiş)', fontsize=18, fontweight='bold', color='darkgreen')
    ax2.axis('off')
    plt.suptitle('Roma Kolon Dijital Restorasyonu', fontsize=20, fontweight='bold')
    plt.tight_layout()
    plt.savefig('before_after.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("→ 'before_after.png' kaydedildi.")
    print(f"Uygulanan adımlar: {result['steps']}")

In [ ]:
# Parametre ayarı
# edge_thresh : Küçük → kenarları daha katı değerlendir (daha üstte keser)
# extra_ratio : Büyük → daha uzun şaft eklenir
# Kesim çok aşağıda kalıyorsa:
# result = restore_column(test_path, edge_thresh=0.15)
# Daha uzun şaft istiyorsan:
# result = restore_column(test_path, extra_ratio=0.80)
print('Parametre ayarı için yorumları kaldır.')

In [ ]:
GOOD_INDICES = [2, 3, 6, 9, 12, 14, 15, 17, 19, 20, 21, 25, 26, 27, 29]
SAVE_DIR     = ('/content/drive/MyDrive/sutun_veriseti_1/restored'
                if IN_COLAB else './restored')
os.makedirs(SAVE_DIR, exist_ok=True)
damaged_files = [f for f in os.listdir(DAMAGED_DIR)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
for idx in GOOD_INDICES:
    if idx >= len(damaged_files): continue
    path = os.path.join(DAMAGED_DIR, damaged_files[idx])
    try:
        res = restore_column(path, extra_ratio=0.55, sym_threshold=0.13)
        if res:
            bgr  = cv2.cvtColor(res['final'], cv2.COLOR_RGB2BGR)
            save = os.path.join(SAVE_DIR, f'restored_{idx:02d}_{damaged_files[idx]}')
            cv2.imwrite(save, bgr)
            print(f'  [{idx}] → {save}')
    except Exception as e:
        print(f'  [{idx}] Hata: {e}')
print('Toplu işleme tamamlandı.')